In [2]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score
from pyvi import ViTokenizer

In [3]:
train_df = pd.read_csv("train_data.csv")
test_df = pd.read_csv("test_data.csv")

# Xử lý các giá trị NaN
train_df['sarcasm_type'] = train_df['sarcasm_type'].fillna("None")
test_df['sarcasm_type'] = test_df['sarcasm_type'].fillna("None")

# CHIẾN LƯỢC CÂN BẰNG DỮ LIỆU (RESAMPLING)
print("Tỉ lệ trước khi cân bằng:\n", train_df['sarcasm_label'].value_counts())

# Tách nhóm
df_non = train_df[train_df['sarcasm_label'] == 'Non-Sarcastic']
df_sar = train_df[train_df['sarcasm_label'] == 'Sarcastic']

# 1. Undersample: Giảm Non-Sarcastic xuống 1500 mẫu
df_non_sampled = df_non.sample(n=1500, random_state=42)

# 2. Oversample: Nhân bản từng Sarcasm Type lên tối thiểu 150 mẫu
oversampled_list = []
for s_type in df_sar['sarcasm_type'].unique():
    df_type = df_sar[df_sar['sarcasm_type'] == s_type]
    target_count = max(150, len(df_type)) # Nếu đã > 150 thì giữ nguyên, nhỏ hơn thì nhân bản
    df_type_sampled = df_type.sample(n=target_count, replace=True, random_state=42)
    oversampled_list.append(df_type_sampled)

df_sar_balanced = pd.concat(oversampled_list)

# 3. Gộp lại và xáo trộn ngẫu nhiên
train_df = pd.concat([df_non_sampled, df_sar_balanced]).sample(frac=1.0, random_state=42).reset_index(drop=True)
print("Tỉ lệ sau khi cân bằng:\n", train_df['sarcasm_type'].value_counts())

# TẠO NHÃN 6 CLASS CHO BÀI TOÁN PHÂN LOẠI
def get_6_labels(row):
    if row['sarcasm_label'] == 'Non-Sarcastic': return 'Non-Sarcastic'
    if pd.isna(row['sarcasm_type']) or row['sarcasm_type'] == 'None': return 'Propositional Contradiction'
    return row['sarcasm_type']

train_df['final_label'] = train_df.apply(get_6_labels, axis=1)
test_df['final_label'] = test_df.apply(get_6_labels, axis=1)

# Mã hóa nhãn chữ thành số (0 -> 5)
le = LabelEncoder()
train_df['label'] = le.fit_transform(train_df['final_label'])
test_df['label'] = le.transform(test_df['final_label'])

Tỉ lệ trước khi cân bằng:
 sarcasm_label
Non-Sarcastic    6639
Sarcastic         523
Name: count, dtype: int64
Tỉ lệ sau khi cân bằng:
 sarcasm_type
None                           1500
Propositional Contradiction     180
Lexical Contradiction           171
Hypothetical                    150
Rhetorical Question             150
Wh-Question                     150
Name: count, dtype: int64


In [4]:
def segment_text(text):
    return ViTokenizer.tokenize(str(text))

# Tách từ riêng rẽ cho Video và Comment
train_df['video_seg'] = train_df['video_core_content'].apply(segment_text)
train_df['comment_seg'] = train_df['comment'].apply(segment_text)

test_df['video_seg'] = test_df['video_core_content'].apply(segment_text)
test_df['comment_seg'] = test_df['comment'].apply(segment_text)

# Đưa vào Dataset
train_ds = Dataset.from_pandas(train_df[['video_seg', 'comment_seg', 'label']])
test_ds = Dataset.from_pandas(test_df[['video_seg', 'comment_seg', 'label']])

model_name = "vinai/phobert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # Truyền tách biệt video và comment để Tokenizer hiểu 2 vế
    return tokenizer(
        examples["video_seg"], 
        examples["comment_seg"], 
        padding="max_length", 
        truncation="only_first", # BẢO VỆ COMMENT KHÔNG BỊ CẮT
        max_length=256
    )

train_tokenized = train_ds.map(tokenize_function, batched=True)
test_tokenized = test_ds.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 45295.36it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# 3. ĐÁNH GIÁ VÀ HUẤN LUYỆN
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"f1_macro": f1_score(labels, predictions, average='macro')}

training_args = TrainingArguments(
    output_dir="./phobert_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16, 
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=True, 
    warmup_ratio=0.1,
)

trainer = Trainer(
    model=model, 
    args=training_args, 
    train_dataset=train_tokenized, 
    eval_dataset=test_tokenized, 
    compute_metrics=compute_metrics,
)

print("⏳ Bắt đầu quá trình Train...")
trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


⏳ Bắt đầu quá trình Train...


Epoch,Training Loss,Validation Loss,F1 Macro
1,No log,0.506032,0.160365
2,No log,0.432966,0.194839
3,No log,0.389306,0.271575
4,0.925072,0.368463,0.359659
5,0.925072,0.392721,0.419855


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.75it/s]


TrainOutput(global_step=720, training_loss=0.8113340059916179, metrics={'train_runtime': 184.9382, 'train_samples_per_second': 62.21, 'train_steps_per_second': 3.893, 'total_flos': 1513600704046080.0, 'train_loss': 0.8113340059916179, 'epoch': 5.0})

In [6]:
print("📊 Đang xuất báo cáo và lưu file...")
predictions = trainer.predict(test_tokenized)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

target_names = le.inverse_transform([0, 1, 2, 3, 4, 5])

print("="*60)
print("🏆 BÁO CÁO KẾT QUẢ PHO-BERT")
print("="*60)
print(classification_report(y_true, y_pred, target_names=target_names))

📊 Đang xuất báo cáo và lưu file...


🏆 BÁO CÁO KẾT QUẢ PHO-BERT
                             precision    recall  f1-score   support

               Hypothetical       0.24      0.33      0.28        12
      Lexical Contradiction       0.43      0.16      0.23        19
              Non-Sarcastic       0.97      0.94      0.96       738
Propositional Contradiction       0.22      0.35      0.27        20
        Rhetorical Question       0.29      1.00      0.45         5
                Wh-Question       0.25      0.50      0.33         2

                   accuracy                           0.90       796
                  macro avg       0.40      0.55      0.42       796
               weighted avg       0.92      0.90      0.91       796



In [7]:
predicted_merged = le.inverse_transform(y_pred)

# Tách ngược lại thành Label và Type để chuẩn format
test_df['predicted_label'] = ['Non-Sarcastic' if p == 'Non-Sarcastic' else 'Sarcastic' for p in predicted_merged]
test_df['predicted_type'] = ['None' if p == 'Non-Sarcastic' else p for p in predicted_merged]

output_file = "test_result_PhoBERT.csv"
test_df.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"✅ Đã lưu kết quả chi tiết của PhoBERT vào: {output_file}")

✅ Đã lưu kết quả chi tiết của PhoBERT vào: test_result_PhoBERT.csv
